# Week 9 — Agentic Copilot (Full Pipeline)

**Learning Goals:**
- Wire predict → explain → retrieve → recommend into a single agent pipeline
- Implement safety guardrails (abstention, escalation)
- Test with fleet-level batch queries

**Deliverables:**
- End-to-end `OpsCopilot.analyze_engine()` demo
- `get_fleet_summary()` across all test engines
- Formatted maintenance report

In [ ]:
import sys, torch
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from copilot.agent import OpsCopilot
from copilot.tools import predict_rul, explain_prediction, retrieve_knowledge, recommend_action
from data_loader import load_test, load_train
from preprocess import preprocess_pipeline

print(f'PyTorch {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load Model + Data

In [ ]:
# ---- Load best checkpoint from training ----
# Adjust the path to your best saved model
from models.cnn_transformer import CNNTransformerModel
from infer import load_model

CHECKPOINT = '../checkpoints/cnn_transformer_best.pt'  # adjust as needed
WINDOW = 30
N_FEATURES = 14

model = CNNTransformerModel(
    n_features=N_FEATURES, seq_len=WINDOW,
    cnn_channels=64, d_model=128, nhead=4, num_layers=2,
    dropout=0.2
)

# TODO: load weights if checkpoint exists, else skip
import os
if os.path.exists(CHECKPOINT):
    state = torch.load(CHECKPOINT, map_location=device)
    model.load_state_dict(state['model_state_dict'] if 'model_state_dict' in state else state)
    print(f'Loaded checkpoint from {CHECKPOINT}')
else:
    print(f'No checkpoint at {CHECKPOINT} — using random weights for demo')

model.to(device)
model.eval()
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Prepare test data
test_df, rul_true = load_test('../data', 'FD001')
train_df = load_train('../data', 'FD001')

result = preprocess_pipeline(train_df, test_df, window_size=WINDOW)
X_test = result['X_test']

print(f'Test windows: {X_test.shape}')
print(f'True RULs: {rul_true.shape}')

## 2. Single-Engine Copilot Demo

In [ ]:
copilot = OpsCopilot(
    model=model,
    kb_dir='../kb',
    device=device,
    escalation_threshold=20.0,
)

# Analyze engine 0
engine_id = 0
x_input = torch.tensor(X_test[engine_id:engine_id+1], dtype=torch.float32).to(device)

report = copilot.analyze_engine(
    x_input,
    engine_id=engine_id,
    true_rul=rul_true[engine_id],
    n_mc_samples=50,
)

# Print formatted
formatted = copilot.format_report(report)
print(formatted)

## 3. Safety Checks: Abstention & Escalation

In [ ]:
# Test abstention on engines with high uncertainty
abstention_count = 0
escalation_count = 0
normal_count = 0

for i in range(min(len(X_test), 100)):
    x = torch.tensor(X_test[i:i+1], dtype=torch.float32).to(device)
    result = copilot.analyze_engine(x, engine_id=i, n_mc_samples=30)
    
    action = result.get('recommendation', {}).get('action', 'unknown')
    if action == 'escalate_to_human':
        escalation_count += 1
    elif action == 'abstain':
        abstention_count += 1
    else:
        normal_count += 1

print(f'Normal recommendations: {normal_count}')
print(f'Escalated to human:     {escalation_count}')
print(f'Abstained:              {abstention_count}')

## 4. Fleet-Level Batch Analysis

In [ ]:
# Batch analyze all test engines
X_all = torch.tensor(X_test, dtype=torch.float32).to(device)
fleet_results = copilot.batch_analyze(X_all, n_mc_samples=30)
summary = copilot.get_fleet_summary(fleet_results)

print('=== Fleet Summary ===')
for k, v in summary.items():
    print(f'  {k}: {v}')

In [ ]:
# Visualize fleet RUL distribution
ruls = [r['prediction']['rul_mean'] for r in fleet_results]
stds = [r['prediction']['rul_std'] for r in fleet_results]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(ruls, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(x=30, color='red', ls='--', label='Urgent threshold (30)')
axes[0].set_xlabel('Predicted RUL (cycles)')
axes[0].set_ylabel('Count')
axes[0].set_title('Fleet RUL Distribution')
axes[0].legend()

axes[1].scatter(ruls, stds, alpha=0.6, c='steelblue')
axes[1].axhline(y=20, color='red', ls='--', label='Escalation threshold')
axes[1].set_xlabel('Predicted RUL')
axes[1].set_ylabel('Uncertainty (std)')
axes[1].set_title('RUL vs Uncertainty')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/fleet_rul_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Full Maintenance Report

In [ ]:
# Print top-5 most urgent engines
urgent = sorted(fleet_results, key=lambda r: r['prediction']['rul_mean'])[:5]

for r in urgent:
    print(copilot.format_report(r))
    print('=' * 60)